# Your AI Is Only as Good as Your Data Foundation
## Summit Gear Co. — Live Demo
### AI Camp / Qlik Event — October 1, 2026 | Holladay, UT

**Speaker:** Todd Racker, Snowflake

**Nick Magnuson** (Qlik Head of AI) presents: *What is the role of Dashboards in the world of GenAI self-service analytics?*

**Todd's talk:** The data foundation that makes AI trustworthy — from messy streaming data to a governed Cortex Agent in 30 minutes.

---

### Architecture

```
Java Streaming App (Snowpipe Streaming SDK)
        |
        v
  BRONZE (Raw Landing)          →  RAW_POS, RAW_ECOM, RAW_INVENTORY, RAW_REVIEWS
        |  Dynamic Tables (auto-refresh)
        v
  SILVER (Cleaned + Conformed)  →  ORDERS, INVENTORY, REVIEWS (AI Sentiment), CUSTOMERS
        |  Dynamic Tables
        v
  GOLD (Business Aggregates)    →  STORE_PERFORMANCE, PRODUCT_ANALYTICS, INVENTORY_HEALTH
        |  Governance (RBAC, Masking, Row Access)
        v
  SEMANTIC VIEW + CORTEX AGENT  →  Natural Language Q&A
        |
        v
  AI Consumers: Cortex Agent, Qlik Dashboards, Apps
```

### Pre-Demo Setup

1. Clone this repo and configure `summit-gear-streamer/profile.json`
2. Compile: `cd summit-gear-streamer && mvn compile`
3. Run the cells below in order to create all Snowflake objects
4. Pre-seed: run streamer 5-10 min, stop, restart on stage


---
## 1. Database & Schemas

In [ ]:
CREATE DATABASE IF NOT EXISTS AI_CAMP;
CREATE SCHEMA IF NOT EXISTS AI_CAMP.BRONZE;
CREATE SCHEMA IF NOT EXISTS AI_CAMP.SILVER;
CREATE SCHEMA IF NOT EXISTS AI_CAMP.GOLD;
CREATE SCHEMA IF NOT EXISTS AI_CAMP.GOVERNANCE;
CREATE SCHEMA IF NOT EXISTS AI_CAMP.SEMANTIC;

---
## 2. Bronze Landing Tables

Raw landing tables for the Java Snowpipe Streaming app. Date columns are STRING to capture messy source formats.

In [ ]:
-- POS transactions (messy: duplicate IDs, null store_ids, product name typos, mixed date formats)
CREATE OR REPLACE TABLE AI_CAMP.BRONZE.RAW_POS_TRANSACTIONS (
    transaction_id STRING, store_id STRING, product_name STRING, sku STRING,
    quantity NUMBER, unit_price FLOAT, total_amount FLOAT, payment_method STRING,
    customer_email STRING, transaction_date STRING,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- E-commerce orders (messy: inconsistent status values, missing coupons)
CREATE OR REPLACE TABLE AI_CAMP.BRONZE.RAW_ECOMMERCE_ORDERS (
    order_id STRING, customer_email STRING, product_name STRING, sku STRING,
    quantity NUMBER, unit_price FLOAT, subtotal FLOAT, shipping_cost FLOAT,
    coupon_code STRING, status STRING, shipping_address STRING, billing_address STRING,
    order_timestamp STRING,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Inventory feed (messy: negative quantities, missing dates, orphan SKUs)
CREATE OR REPLACE TABLE AI_CAMP.BRONZE.RAW_INVENTORY_FEED (
    store_id STRING, sku STRING, product_name STRING, quantity_on_hand NUMBER,
    snapshot_date STRING,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Customer reviews (messy: garbage text, varied date formats)
CREATE OR REPLACE TABLE AI_CAMP.BRONZE.RAW_CUSTOMER_REVIEWS (
    review_id STRING, product_sku STRING, customer_name STRING, star_rating NUMBER,
    review_text STRING, review_date STRING,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

### Reference Data: 12 Utah Store Locations

In [ ]:
CREATE OR REPLACE TABLE AI_CAMP.BRONZE.STORE_LOCATIONS (
    store_id STRING, store_name STRING, city STRING, region STRING,
    address STRING, manager_name STRING, opened_date DATE
);

INSERT INTO AI_CAMP.BRONZE.STORE_LOCATIONS VALUES
('STORE-001', 'Summit Gear Park City',     'Park City',          'Wasatch North',    '1820 Park Ave, Park City, UT 84060',                'Jake Anderson',     '2019-03-15'),
('STORE-002', 'Summit Gear Draper',         'Draper',             'Salt Lake South',  '12300 S Lone Peak Pkwy, Draper, UT 84020',          'Mia Thompson',      '2020-06-01'),
('STORE-003', 'Summit Gear Lehi',           'Lehi',               'Utah Valley',      '3500 Digital Dr, Lehi, UT 84043',                   'Ethan Rivera',      '2021-01-10'),
('STORE-004', 'Summit Gear Ogden',          'Ogden',              'Wasatch North',    '2450 Washington Blvd, Ogden, UT 84401',             'Chloe Nguyen',      '2019-09-20'),
('STORE-005', 'Summit Gear Provo',          'Provo',              'Utah Valley',      '1200 N University Ave, Provo, UT 84604',            'Dylan Mortensen',   '2020-03-01'),
('STORE-006', 'Summit Gear Sandy',          'Sandy',              'Salt Lake South',  '9400 S State St, Sandy, UT 84070',                  'Alyssa Christensen','2018-11-15'),
('STORE-007', 'Summit Gear SLC Downtown',   'Salt Lake City',     'Salt Lake Metro',  '350 S Main St, Salt Lake City, UT 84101',           'Caleb Jensen',      '2017-05-20'),
('STORE-008', 'Summit Gear Cottonwood',     'Cottonwood Heights', 'Salt Lake South',  '6910 S Highland Dr, Cottonwood Heights, UT 84121',  'Hailey Garcia',     '2021-07-04'),
('STORE-009', 'Summit Gear Layton',         'Layton',             'Wasatch North',    '765 W Heritage Park Blvd, Layton, UT 84041',        'Brayden Smith',     '2022-02-14'),
('STORE-010', 'Summit Gear Orem',           'Orem',               'Utah Valley',      '580 E University Pkwy, Orem, UT 84097',             'Kylie Peterson',    '2020-08-10'),
('STORE-011', 'Summit Gear Logan',          'Logan',              'Cache Valley',     '1400 N Main St, Logan, UT 84341',                   'Tanner Olsen',      '2023-03-01'),
('STORE-012', 'Summit Gear St George',      'St. George',         'Southern Utah',    '250 N Red Cliffs Dr, St. George, UT 84790',         'Sierra Johnson',    '2022-10-15');

### Reference Data: 34 Products (Ski, Bike, Hike, Climb, Run, Camp)

In [ ]:
CREATE OR REPLACE TABLE AI_CAMP.BRONZE.PRODUCT_CATALOG (
    sku STRING, product_name STRING, category STRING, subcategory STRING,
    brand STRING, msrp FLOAT, cost FLOAT, season STRING
);

INSERT INTO AI_CAMP.BRONZE.PRODUCT_CATALOG VALUES
('SKU-1001', 'Men''s Powder Down Jacket',  'Ski',   'Jackets',        'Summit Alpine',  349.99, 140.00, 'Winter'),
('SKU-1002', 'Women''s Backcountry Shell',  'Ski',   'Jackets',        'Summit Alpine',  429.99, 172.00, 'Winter'),
('SKU-1003', 'All-Mountain Ski 170cm',      'Ski',   'Skis',           'Wasatch Works',  699.99, 280.00, 'Winter'),
('SKU-1004', 'Freeride Ski Boot',           'Ski',   'Boots',          'Peak Pursuit',   549.99, 220.00, 'Winter'),
('SKU-1005', 'Insulated Ski Gloves',        'Ski',   'Accessories',    'Summit Alpine',   79.99,  32.00, 'Winter'),
('SKU-1006', 'MIPS Ski Helmet',             'Ski',   'Helmets',        'Peak Pursuit',   199.99,  80.00, 'Winter'),
('SKU-1007', 'Ski Goggle Pro UV',           'Ski',   'Accessories',    'Summit Alpine',  159.99,  64.00, 'Winter'),
('SKU-1008', 'Youth Ski Package',           'Ski',   'Packages',       'Wasatch Works',  399.99, 160.00, 'Winter'),
('SKU-2001', 'Carbon Trail MTB 29er',       'Bike',  'Mountain Bikes', 'Zion Cycles',   2899.99,1160.00, 'Summer'),
('SKU-2002', 'Enduro Full Suspension',      'Bike',  'Mountain Bikes', 'Zion Cycles',   3499.99,1400.00, 'Summer'),
('SKU-2003', 'Gravel Adventure Bike',       'Bike',  'Gravel Bikes',   'Zion Cycles',   1899.99, 760.00, 'Year-Round'),
('SKU-2004', 'MTB Clipless Pedals',         'Bike',  'Components',     'Peak Pursuit',    129.99,  52.00, 'Summer'),
('SKU-2005', 'Full Face MTB Helmet',        'Bike',  'Helmets',        'Peak Pursuit',    249.99, 100.00, 'Summer'),
('SKU-2006', 'Bike Repair Multi-Tool',      'Bike',  'Accessories',    'Summit Alpine',    34.99,  14.00, 'Year-Round'),
('SKU-3001', 'Ultralight 55L Pack',         'Hike',  'Backpacks',      'Summit Alpine',   279.99, 112.00, 'Summer'),
('SKU-3002', 'Women''s Hiking Boot GTX',    'Hike',  'Footwear',       'Peak Pursuit',    219.99,  88.00, 'Year-Round'),
('SKU-3003', 'Men''s Trail Runner',         'Hike',  'Footwear',       'Peak Pursuit',    149.99,  60.00, 'Summer'),
('SKU-3004', 'Trekking Poles Carbon',       'Hike',  'Accessories',    'Summit Alpine',    99.99,  40.00, 'Year-Round'),
('SKU-3005', '3-Season 2P Tent',            'Hike',  'Shelter',        'Summit Alpine',   349.99, 140.00, 'Summer'),
('SKU-3006', 'Hydration Reservoir 3L',      'Hike',  'Accessories',    'Summit Alpine',    39.99,  16.00, 'Year-Round'),
('SKU-4001', 'Dynamic Climbing Rope 60m',   'Climb', 'Ropes',          'Wasatch Works',   199.99,  80.00, 'Year-Round'),
('SKU-4002', 'Climbing Harness Sport',      'Climb', 'Harnesses',      'Wasatch Works',    89.99,  36.00, 'Year-Round'),
('SKU-4003', 'Quickdraw Set 6-Pack',        'Climb', 'Protection',     'Wasatch Works',    79.99,  32.00, 'Year-Round'),
('SKU-4004', 'Climbing Shoes Performance',  'Climb', 'Footwear',       'Peak Pursuit',    169.99,  68.00, 'Year-Round'),
('SKU-4005', 'Chalk Bag with Belt',         'Climb', 'Accessories',    'Summit Alpine',    24.99,  10.00, 'Year-Round'),
('SKU-5001', 'Trail Running Vest 8L',       'Run',   'Packs',          'Summit Alpine',   139.99,  56.00, 'Year-Round'),
('SKU-5002', 'GPS Running Watch',           'Run',   'Electronics',    'Peak Pursuit',    399.99, 160.00, 'Year-Round'),
('SKU-5003', 'Running Headlamp 500 Lumen',  'Run',   'Accessories',    'Summit Alpine',    49.99,  20.00, 'Year-Round'),
('SKU-6001', '4-Season Expedition Tent',    'Camp',  'Shelter',        'Summit Alpine',   599.99, 240.00, 'Winter'),
('SKU-6002', 'Down Sleeping Bag 0F',        'Camp',  'Sleep Systems',  'Summit Alpine',   449.99, 180.00, 'Winter'),
('SKU-6003', 'Ultralight Sleeping Pad',     'Camp',  'Sleep Systems',  'Summit Alpine',   179.99,  72.00, 'Year-Round'),
('SKU-6004', 'Camp Stove Compact',          'Camp',  'Cooking',        'Wasatch Works',    49.99,  20.00, 'Year-Round'),
('SKU-6005', 'Bear Canister',               'Camp',  'Storage',        'Wasatch Works',    79.99,  32.00, 'Summer'),
('SKU-6006', 'Collapsible Water Filter',    'Camp',  'Hydration',      'Summit Alpine',    44.99,  18.00, 'Year-Round');

---
## DEMO ACT 1: "The Mess"

> *"This is what most companies' data actually looks like. And this is what your AI would be answering questions from."*

Start the Java streamer in a terminal:
```bash
cd summit-gear-streamer && mvn exec:java
```

In [ ]:
-- Show messy POS data: typos, null store_ids, mixed date formats
SELECT transaction_id, store_id, product_name, sku, transaction_date
FROM AI_CAMP.BRONZE.RAW_POS_TRANSACTIONS LIMIT 20;

In [ ]:
-- Show messy e-commerce data: inconsistent status values
SELECT order_id, status, order_timestamp
FROM AI_CAMP.BRONZE.RAW_ECOMMERCE_ORDERS LIMIT 20;

In [ ]:
-- Data is flowing in real-time!
SELECT 'POS' AS source, COUNT(*) AS row_count FROM AI_CAMP.BRONZE.RAW_POS_TRANSACTIONS
UNION ALL SELECT 'Ecommerce', COUNT(*) FROM AI_CAMP.BRONZE.RAW_ECOMMERCE_ORDERS
UNION ALL SELECT 'Inventory', COUNT(*) FROM AI_CAMP.BRONZE.RAW_INVENTORY_FEED
UNION ALL SELECT 'Reviews', COUNT(*) FROM AI_CAMP.BRONZE.RAW_CUSTOMER_REVIEWS;

---
## 3. Silver Dynamic Tables — Transformation

> *"Dynamic tables automatically clean and transform the data. 1-minute lag, no scheduling, no orchestration."*

In [ ]:
-- Unified ORDERS: merges POS + ecommerce, deduplicates, standardizes dates, joins to catalog
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.SILVER.ORDERS
    TARGET_LAG = '1 minute'
    WAREHOUSE = DEMO_WH
AS
WITH pos_cleaned AS (
    SELECT
        transaction_id AS order_id, store_id, product_name AS raw_product_name, sku,
        quantity, unit_price, total_amount, payment_method, customer_email,
        TRY_TO_TIMESTAMP_NTZ(transaction_date) AS order_timestamp,
        'IN_STORE' AS channel, loaded_at
    FROM AI_CAMP.BRONZE.RAW_POS_TRANSACTIONS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY loaded_at DESC) = 1
),
ecom_cleaned AS (
    SELECT
        order_id, NULL AS store_id, product_name AS raw_product_name, sku,
        quantity, unit_price, subtotal + COALESCE(shipping_cost, 0) AS total_amount,
        'Online' AS payment_method, customer_email,
        TRY_TO_TIMESTAMP_NTZ(order_timestamp) AS order_timestamp,
        'ONLINE' AS channel, loaded_at
    FROM AI_CAMP.BRONZE.RAW_ECOMMERCE_ORDERS
),
unioned AS (
    SELECT * FROM pos_cleaned UNION ALL SELECT * FROM ecom_cleaned
)
SELECT
    u.order_id, u.store_id, s.store_name, s.city AS store_city, s.region AS store_region,
    COALESCE(p.product_name, u.raw_product_name) AS product_name,
    u.sku, p.category AS product_category, p.subcategory AS product_subcategory, p.brand,
    u.quantity, u.unit_price, u.total_amount, u.payment_method, u.customer_email,
    u.order_timestamp, u.order_timestamp::DATE AS order_date, u.channel
FROM unioned u
LEFT JOIN AI_CAMP.BRONZE.PRODUCT_CATALOG p ON u.sku = p.sku
LEFT JOIN AI_CAMP.BRONZE.STORE_LOCATIONS s ON u.store_id = s.store_id;

In [ ]:
-- Cleaned INVENTORY: filters negatives, validates SKUs, parses dates
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.SILVER.INVENTORY
    TARGET_LAG = '1 minute'
    WAREHOUSE = DEMO_WH
AS
SELECT i.store_id, s.store_name, s.city AS store_city, s.region AS store_region,
    i.sku, COALESCE(p.product_name, i.product_name) AS product_name,
    p.category AS product_category, p.subcategory AS product_subcategory,
    i.quantity_on_hand, TRY_TO_DATE(i.snapshot_date) AS snapshot_date, i.loaded_at
FROM AI_CAMP.BRONZE.RAW_INVENTORY_FEED i
LEFT JOIN AI_CAMP.BRONZE.PRODUCT_CATALOG p ON i.sku = p.sku
LEFT JOIN AI_CAMP.BRONZE.STORE_LOCATIONS s ON i.store_id = s.store_id
WHERE i.quantity_on_hand >= 0 AND i.snapshot_date IS NOT NULL
  AND TRY_TO_DATE(i.snapshot_date) IS NOT NULL AND p.sku IS NOT NULL;

In [ ]:
-- REVIEWS with AI-powered sentiment scoring using Snowflake Cortex
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.SILVER.REVIEWS
    TARGET_LAG = '5 minutes'
    WAREHOUSE = DEMO_WH
AS
SELECT r.review_id, r.product_sku, p.product_name, p.category AS product_category,
    r.customer_name, r.star_rating, r.review_text,
    SNOWFLAKE.CORTEX.SENTIMENT(r.review_text) AS sentiment_score,
    TRY_TO_DATE(r.review_date) AS review_date, r.loaded_at
FROM AI_CAMP.BRONZE.RAW_CUSTOMER_REVIEWS r
LEFT JOIN AI_CAMP.BRONZE.PRODUCT_CATALOG p ON r.product_sku = p.sku
WHERE r.review_text IS NOT NULL AND LENGTH(TRIM(r.review_text)) > 10;

In [ ]:
-- Deduplicated CUSTOMERS
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.SILVER.CUSTOMERS
    TARGET_LAG = '5 minutes'
    WAREHOUSE = DEMO_WH
AS
WITH all_emails AS (
    SELECT customer_email, loaded_at FROM AI_CAMP.BRONZE.RAW_POS_TRANSACTIONS WHERE customer_email IS NOT NULL
    UNION ALL
    SELECT customer_email, loaded_at FROM AI_CAMP.BRONZE.RAW_ECOMMERCE_ORDERS WHERE customer_email IS NOT NULL
)
SELECT customer_email, COUNT(*) AS total_orders, MIN(loaded_at) AS first_seen, MAX(loaded_at) AS last_seen
FROM all_emails GROUP BY customer_email;

### DEMO ACT 3: Show the Clean Data

> *"From raw mess to clean, unified, enriched data. All running automatically."*

In [ ]:
-- Show the cleaned, unified orders
SELECT order_id, store_name, product_name, product_category, channel, order_date, total_amount
FROM AI_CAMP.SILVER.ORDERS LIMIT 20;

In [ ]:
-- Show AI sentiment scoring on reviews
SELECT review_text, sentiment_score, star_rating, product_name
FROM AI_CAMP.SILVER.REVIEWS ORDER BY sentiment_score ASC LIMIT 10;

---
## 4. Gold Dynamic Tables — Business Aggregates

In [ ]:
-- STORE_PERFORMANCE: daily revenue, orders, AOV by store/channel/category
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.GOLD.STORE_PERFORMANCE
    TARGET_LAG = '1 minute'
    WAREHOUSE = DEMO_WH
AS
SELECT store_id, store_name, store_city, store_region, order_date, channel, product_category,
    COUNT(*) AS total_orders, SUM(total_amount) AS revenue, SUM(quantity) AS units_sold,
    ROUND(AVG(total_amount), 2) AS avg_order_value, COUNT(DISTINCT customer_email) AS unique_customers
FROM AI_CAMP.SILVER.ORDERS WHERE order_date IS NOT NULL GROUP BY ALL;

In [ ]:
-- PRODUCT_ANALYTICS: sales + sentiment per product
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.GOLD.PRODUCT_ANALYTICS
    TARGET_LAG = '5 minutes'
    WAREHOUSE = DEMO_WH
AS
SELECT p.sku, p.product_name, p.category, p.subcategory, p.brand, p.season, p.msrp, p.cost,
    COALESCE(o.total_orders, 0) AS total_orders, COALESCE(o.units_sold, 0) AS units_sold,
    COALESCE(o.revenue, 0) AS revenue,
    ROUND(COALESCE(o.revenue, 0) - (COALESCE(o.units_sold, 0) * p.cost), 2) AS gross_margin,
    r.avg_sentiment, r.avg_star_rating, COALESCE(r.review_count, 0) AS review_count
FROM AI_CAMP.BRONZE.PRODUCT_CATALOG p
LEFT JOIN (SELECT sku, COUNT(*) AS total_orders, SUM(quantity) AS units_sold, SUM(total_amount) AS revenue
           FROM AI_CAMP.SILVER.ORDERS GROUP BY sku) o ON p.sku = o.sku
LEFT JOIN (SELECT product_sku, ROUND(AVG(sentiment_score), 4) AS avg_sentiment,
                  ROUND(AVG(star_rating), 2) AS avg_star_rating, COUNT(*) AS review_count
           FROM AI_CAMP.SILVER.REVIEWS GROUP BY product_sku) r ON p.sku = r.product_sku;

In [ ]:
-- INVENTORY_HEALTH: current stock levels with low/out-of-stock flags
CREATE OR REPLACE DYNAMIC TABLE AI_CAMP.GOLD.INVENTORY_HEALTH
    TARGET_LAG = '5 minutes'
    WAREHOUSE = DEMO_WH
AS
SELECT i.store_id, i.store_name, i.store_city, i.store_region,
    i.sku, i.product_name, i.product_category, i.product_subcategory,
    i.quantity_on_hand, i.snapshot_date,
    CASE WHEN i.quantity_on_hand < 5 THEN TRUE ELSE FALSE END AS low_stock_flag,
    CASE WHEN i.quantity_on_hand = 0 THEN TRUE ELSE FALSE END AS out_of_stock_flag
FROM AI_CAMP.SILVER.INVENTORY i
QUALIFY ROW_NUMBER() OVER (PARTITION BY i.store_id, i.sku ORDER BY i.snapshot_date DESC, i.loaded_at DESC) = 1;

In [ ]:
-- Show gold aggregates
SELECT store_name, store_region, SUM(revenue) AS total_revenue, SUM(units_sold) AS total_units
FROM AI_CAMP.GOLD.STORE_PERFORMANCE GROUP BY store_name, store_region ORDER BY total_revenue DESC;

---
## 5. Governance & RBAC

> *"Clean data isn't enough. It has to be governed. Who can see what?"*

In [ ]:
-- Create roles
CREATE ROLE IF NOT EXISTS SUMMIT_ANALYST;
CREATE ROLE IF NOT EXISTS SUMMIT_STORE_MGR_PARK_CITY;
GRANT ROLE SUMMIT_ANALYST TO USER tracker;
GRANT ROLE SUMMIT_STORE_MGR_PARK_CITY TO USER tracker;

-- Grant access
GRANT USAGE ON WAREHOUSE DEMO_WH TO ROLE SUMMIT_ANALYST;
GRANT USAGE ON WAREHOUSE DEMO_WH TO ROLE SUMMIT_STORE_MGR_PARK_CITY;
GRANT USAGE ON DATABASE AI_CAMP TO ROLE SUMMIT_ANALYST;
GRANT USAGE ON DATABASE AI_CAMP TO ROLE SUMMIT_STORE_MGR_PARK_CITY;
GRANT USAGE ON ALL SCHEMAS IN DATABASE AI_CAMP TO ROLE SUMMIT_ANALYST;
GRANT USAGE ON ALL SCHEMAS IN DATABASE AI_CAMP TO ROLE SUMMIT_STORE_MGR_PARK_CITY;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA AI_CAMP.GOLD TO ROLE SUMMIT_ANALYST;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA AI_CAMP.GOLD TO ROLE SUMMIT_STORE_MGR_PARK_CITY;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA AI_CAMP.SILVER TO ROLE SUMMIT_ANALYST;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA AI_CAMP.SILVER TO ROLE SUMMIT_STORE_MGR_PARK_CITY;

In [ ]:
-- PII tagging
CREATE TAG IF NOT EXISTS AI_CAMP.GOVERNANCE.PII_TYPE ALLOWED_VALUES 'EMAIL', 'ADDRESS', 'NAME';
ALTER TABLE AI_CAMP.SILVER.ORDERS ALTER COLUMN customer_email SET TAG AI_CAMP.GOVERNANCE.PII_TYPE = 'EMAIL';
ALTER TABLE AI_CAMP.SILVER.REVIEWS ALTER COLUMN customer_name SET TAG AI_CAMP.GOVERNANCE.PII_TYPE = 'NAME';

In [ ]:
-- Column masking: analysts see full data, store managers see masked PII
CREATE OR REPLACE MASKING POLICY AI_CAMP.GOVERNANCE.MASK_EMAIL AS (val STRING) RETURNS STRING ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SUMMIT_ANALYST') THEN val
        ELSE REGEXP_REPLACE(val, '(^[a-zA-Z0-9]{1})([^@]*)(@.*)', '\\1****\\3')
    END;

CREATE OR REPLACE MASKING POLICY AI_CAMP.GOVERNANCE.MASK_NAME AS (val STRING) RETURNS STRING ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SUMMIT_ANALYST') THEN val
        ELSE CONCAT(LEFT(val, 1), '****')
    END;

ALTER TABLE AI_CAMP.SILVER.ORDERS ALTER COLUMN customer_email SET MASKING POLICY AI_CAMP.GOVERNANCE.MASK_EMAIL;
ALTER TABLE AI_CAMP.SILVER.REVIEWS ALTER COLUMN customer_name SET MASKING POLICY AI_CAMP.GOVERNANCE.MASK_NAME;

In [ ]:
-- Row access policy: store managers only see their store's data
CREATE OR REPLACE ROW ACCESS POLICY AI_CAMP.GOVERNANCE.STORE_ROW_FILTER AS (store_city STRING) RETURNS BOOLEAN ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SUMMIT_ANALYST') THEN TRUE
        WHEN CURRENT_ROLE() = 'SUMMIT_STORE_MGR_PARK_CITY' AND store_city = 'Park City' THEN TRUE
        ELSE FALSE
    END;

ALTER DYNAMIC TABLE AI_CAMP.GOLD.STORE_PERFORMANCE ADD ROW ACCESS POLICY AI_CAMP.GOVERNANCE.STORE_ROW_FILTER ON (store_city);
ALTER DYNAMIC TABLE AI_CAMP.GOLD.INVENTORY_HEALTH ADD ROW ACCESS POLICY AI_CAMP.GOVERNANCE.STORE_ROW_FILTER ON (store_city);

### DEMO ACT 4: Role Switching on Stage

> *"Same table, different views. Every AI tool inherits these controls automatically."*

In [ ]:
-- As ACCOUNTADMIN: full access, full emails
USE ROLE ACCOUNTADMIN;
SELECT customer_email FROM AI_CAMP.SILVER.ORDERS LIMIT 5;

In [ ]:
-- As store manager: masked emails!
USE ROLE SUMMIT_STORE_MGR_PARK_CITY;
SELECT customer_email FROM AI_CAMP.SILVER.ORDERS LIMIT 5;

In [ ]:
-- Row filtering: store manager only sees Park City
USE ROLE SUMMIT_STORE_MGR_PARK_CITY;
SELECT store_name, SUM(revenue) AS revenue
FROM AI_CAMP.GOLD.STORE_PERFORMANCE GROUP BY store_name;

In [ ]:
-- Back to full access: all 12 stores visible
USE ROLE ACCOUNTADMIN;
SELECT store_name, SUM(revenue) AS revenue
FROM AI_CAMP.GOLD.STORE_PERFORMANCE GROUP BY store_name ORDER BY revenue DESC;

---
## 6. Semantic View

> *"The final layer: the Semantic View. This is the contract between your data and every AI consumer. Define metrics, dimensions, and relationships once — every AI tool gets the same truth."*

In [ ]:
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'AI_CAMP.SEMANTIC',
  $$
name: SUMMIT_GEAR_SV
description: >-
  Summit Gear Co. analytics model covering store performance, product analytics
  (including AI-powered sentiment scores), and inventory health across 12 Utah
  retail locations and an e-commerce channel.

tables:
  - name: STORE_PERFORMANCE
    base_table:
      database: AI_CAMP
      schema: GOLD
      table: STORE_PERFORMANCE
    description: >-
      Daily aggregated store performance metrics including revenue, order counts,
      and average order value broken down by store, channel, and product category.
    dimensions:
      - name: STORE_ID
        expr: STORE_ID
        description: Unique store identifier
        data_type: VARCHAR
      - name: STORE_NAME
        expr: STORE_NAME
        description: Store display name (e.g. Summit Gear Park City)
        data_type: VARCHAR
      - name: STORE_CITY
        expr: STORE_CITY
        description: City where the store is located
        data_type: VARCHAR
      - name: STORE_REGION
        expr: STORE_REGION
        description: Geographic region (Wasatch North, Salt Lake South, Utah Valley, Salt Lake Metro, Cache Valley, Southern Utah)
        data_type: VARCHAR
      - name: ORDER_DATE
        expr: ORDER_DATE
        description: Date of the orders
        data_type: DATE
      - name: CHANNEL
        expr: CHANNEL
        description: Sales channel - IN_STORE or ONLINE
        data_type: VARCHAR
      - name: PRODUCT_CATEGORY
        expr: PRODUCT_CATEGORY
        description: Product category (Ski, Bike, Hike, Climb, Run, Camp)
        data_type: VARCHAR
    metrics:
      - name: TOTAL_ORDERS
        expr: SUM(TOTAL_ORDERS)
        description: Total number of orders
      - name: REVENUE
        expr: SUM(REVENUE)
        description: Total revenue in USD
      - name: UNITS_SOLD
        expr: SUM(UNITS_SOLD)
        description: Total units sold
      - name: AVG_ORDER_VALUE
        expr: AVG(AVG_ORDER_VALUE)
        description: Average order value in USD
      - name: UNIQUE_CUSTOMERS
        expr: SUM(UNIQUE_CUSTOMERS)
        description: Count of unique customers

  - name: PRODUCT_ANALYTICS
    base_table:
      database: AI_CAMP
      schema: GOLD
      table: PRODUCT_ANALYTICS
    description: >-
      Product-level analytics combining sales performance with customer sentiment
      from AI-analyzed reviews. One row per product (SKU).
    dimensions:
      - name: SKU
        expr: SKU
        description: Stock keeping unit - unique product identifier
        data_type: VARCHAR
      - name: PRODUCT_NAME
        expr: PRODUCT_NAME
        description: Full product name
        data_type: VARCHAR
      - name: CATEGORY
        expr: CATEGORY
        description: Product category (Ski, Bike, Hike, Climb, Run, Camp)
        data_type: VARCHAR
      - name: SUBCATEGORY
        expr: SUBCATEGORY
        description: Product subcategory (e.g. Jackets, Mountain Bikes, Backpacks)
        data_type: VARCHAR
      - name: BRAND
        expr: BRAND
        description: Product brand (Summit Alpine, Peak Pursuit, Wasatch Works, Zion Cycles)
        data_type: VARCHAR
      - name: SEASON
        expr: SEASON
        description: Primary season for the product (Winter, Summer, Year-Round)
        data_type: VARCHAR
    metrics:
      - name: TOTAL_ORDERS
        expr: SUM(TOTAL_ORDERS)
        description: Total orders for this product
      - name: UNITS_SOLD
        expr: SUM(UNITS_SOLD)
        description: Total units sold
      - name: REVENUE
        expr: SUM(REVENUE)
        description: Total revenue in USD
      - name: GROSS_MARGIN
        expr: SUM(GROSS_MARGIN)
        description: Total gross margin (revenue minus cost of goods)
      - name: AVG_SENTIMENT
        expr: AVG(AVG_SENTIMENT)
        description: Average AI-computed sentiment score from customer reviews (-1 to 1 scale)
      - name: AVG_STAR_RATING
        expr: AVG(AVG_STAR_RATING)
        description: Average customer star rating (1-5)
      - name: REVIEW_COUNT
        expr: SUM(REVIEW_COUNT)
        description: Total number of customer reviews
      - name: MSRP
        expr: AVG(MSRP)
        description: Manufacturer suggested retail price

  - name: INVENTORY_HEALTH
    base_table:
      database: AI_CAMP
      schema: GOLD
      table: INVENTORY_HEALTH
    description: >-
      Current inventory levels per store and product with stock health flags.
      Shows the most recent snapshot for each store-product combination.
    dimensions:
      - name: STORE_ID
        expr: STORE_ID
        description: Store identifier
        data_type: VARCHAR
      - name: STORE_NAME
        expr: STORE_NAME
        description: Store display name
        data_type: VARCHAR
      - name: STORE_CITY
        expr: STORE_CITY
        description: City where the store is located
        data_type: VARCHAR
      - name: STORE_REGION
        expr: STORE_REGION
        description: Geographic region
        data_type: VARCHAR
      - name: SKU
        expr: SKU
        description: Product SKU
        data_type: VARCHAR
      - name: PRODUCT_NAME
        expr: PRODUCT_NAME
        description: Product display name
        data_type: VARCHAR
      - name: PRODUCT_CATEGORY
        expr: PRODUCT_CATEGORY
        description: Product category
        data_type: VARCHAR
      - name: SNAPSHOT_DATE
        expr: SNAPSHOT_DATE
        description: Date of the inventory snapshot
        data_type: DATE
      - name: LOW_STOCK_FLAG
        expr: LOW_STOCK_FLAG
        description: True if quantity on hand is below 5 units
        data_type: BOOLEAN
      - name: OUT_OF_STOCK_FLAG
        expr: OUT_OF_STOCK_FLAG
        description: True if quantity on hand is zero
        data_type: BOOLEAN
    metrics:
      - name: QUANTITY_ON_HAND
        expr: SUM(QUANTITY_ON_HAND)
        description: Total units in stock
      - name: LOW_STOCK_COUNT
        expr: COUNT_IF(LOW_STOCK_FLAG = TRUE)
        description: Number of store-product combinations with low stock
      - name: OUT_OF_STOCK_COUNT
        expr: COUNT_IF(OUT_OF_STOCK_FLAG = TRUE)
        description: Number of store-product combinations that are out of stock

relationships:
  - name: STORE_PERF_TO_PRODUCT
    left_table: STORE_PERFORMANCE
    right_table: PRODUCT_ANALYTICS
    relationship_columns:
      - left_column: PRODUCT_CATEGORY
        right_column: CATEGORY
  - name: INVENTORY_TO_PRODUCT
    left_table: INVENTORY_HEALTH
    right_table: PRODUCT_ANALYTICS
    relationship_columns:
      - left_column: SKU
        right_column: SKU

verified_queries:
  - name: revenue_by_store
    question: What is total revenue by store?
    sql: >-
      SELECT STORE_NAME, SUM(REVENUE) AS TOTAL_REVENUE
      FROM AI_CAMP.GOLD.STORE_PERFORMANCE
      GROUP BY STORE_NAME
      ORDER BY TOTAL_REVENUE DESC
  - name: worst_sentiment_products
    question: Which products have the worst customer sentiment?
    sql: >-
      SELECT PRODUCT_NAME, AVG_SENTIMENT, AVG_STAR_RATING, REVIEW_COUNT
      FROM AI_CAMP.GOLD.PRODUCT_ANALYTICS
      WHERE REVIEW_COUNT > 0
      ORDER BY AVG_SENTIMENT ASC
      LIMIT 10
  - name: low_stock_items
    question: Which products are low on stock across stores?
    sql: >-
      SELECT STORE_NAME, PRODUCT_NAME, QUANTITY_ON_HAND
      FROM AI_CAMP.GOLD.INVENTORY_HEALTH
      WHERE LOW_STOCK_FLAG = TRUE
      ORDER BY QUANTITY_ON_HAND ASC
  - name: avg_order_value_by_channel
    question: What is the average order value by channel?
    sql: >-
      SELECT CHANNEL, SUM(REVENUE) AS TOTAL_REVENUE,
             SUM(TOTAL_ORDERS) AS TOTAL_ORDERS,
             ROUND(AVG(AVG_ORDER_VALUE), 2) AS AVG_ORDER_VALUE
      FROM AI_CAMP.GOLD.STORE_PERFORMANCE
      GROUP BY CHANNEL
  - name: top_categories_by_revenue
    question: What are the top product categories by revenue?
    sql: >-
      SELECT CATEGORY, SUM(REVENUE) AS TOTAL_REVENUE,
             SUM(UNITS_SOLD) AS TOTAL_UNITS,
             ROUND(SUM(GROSS_MARGIN), 2) AS TOTAL_MARGIN
      FROM AI_CAMP.GOLD.PRODUCT_ANALYTICS
      GROUP BY CATEGORY
      ORDER BY TOTAL_REVENUE DESC
  - name: revenue_by_region
    question: How does revenue compare across Utah regions?
    sql: >-
      SELECT STORE_REGION, SUM(REVENUE) AS TOTAL_REVENUE,
             SUM(UNITS_SOLD) AS TOTAL_UNITS
      FROM AI_CAMP.GOLD.STORE_PERFORMANCE
      GROUP BY STORE_REGION
      ORDER BY TOTAL_REVENUE DESC
  $$
);

In [ ]:
-- Inspect the semantic view
DESCRIBE SEMANTIC VIEW AI_CAMP.SEMANTIC.SUMMIT_GEAR_SV;

---
## 7. Cortex Agent

> *"Now the moment of truth. We went from messy CSVs to a full governed foundation. Let's see if the AI can use it."*

In [ ]:
CREATE OR REPLACE AGENT AI_CAMP.SEMANTIC.SUMMIT_GEAR_AGENT
  COMMENT = 'Summit Gear Co. AI assistant'
  PROFILE = '{"display_name": "Summit Gear Assistant", "color": "blue"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  orchestration:
    tool_not_accessible: accept
    budget:
      seconds: 60
      tokens: 16000

  instructions:
    response: "You are the Summit Gear Co. data assistant. Answer questions about store performance, product analytics, inventory health, and customer sentiment across our 12 Utah retail locations and e-commerce channel. Be concise and data-driven."
    orchestration: "Use the Analyst tool for all data questions about revenue, orders, inventory, sentiment, and store performance."
    sample_questions:
      - question: "What is total revenue by store?"
      - question: "Which products have the worst customer sentiment?"
      - question: "Which stores have low inventory?"
      - question: "What is the average order value by channel?"

  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "SummitGearAnalyst"
        description: "Analyzes Summit Gear Co. data including store performance, product analytics with AI sentiment, and inventory health across Utah locations."

  tool_resources:
    SummitGearAnalyst:
      semantic_view: "AI_CAMP.SEMANTIC.SUMMIT_GEAR_SV"
  $$;

In [ ]:
-- Grant access to demo roles
GRANT REFERENCES, SELECT ON SEMANTIC VIEW AI_CAMP.SEMANTIC.SUMMIT_GEAR_SV TO ROLE SUMMIT_ANALYST;
GRANT REFERENCES, SELECT ON SEMANTIC VIEW AI_CAMP.SEMANTIC.SUMMIT_GEAR_SV TO ROLE SUMMIT_STORE_MGR_PARK_CITY;
GRANT USAGE ON AGENT AI_CAMP.SEMANTIC.SUMMIT_GEAR_AGENT TO ROLE SUMMIT_ANALYST;
GRANT USAGE ON AGENT AI_CAMP.SEMANTIC.SUMMIT_GEAR_AGENT TO ROLE SUMMIT_STORE_MGR_PARK_CITY;

### DEMO ACT 6: Ask the Agent Live

Open the agent in **Snowsight > AI & ML > Agents > SUMMIT_GEAR_AGENT** and ask:

1. **"What is total revenue by store?"**
2. **"Which products have the worst customer sentiment?"**
3. **"Which stores have low inventory?"**
4. **"What is the average order value — online vs in-store?"**
5. **Ask the audience for a question!**

> *"This agent can only see what the governance policies allow. If I switched to the store manager role, it could only answer about Park City. The foundation makes the AI trustworthy."*

---
## Close

> *"We went from raw, messy streaming data to a governed AI agent in 30 minutes. That's the data foundation.*
>
> *Ingestion, transformation, governance, semantic layer — each layer makes the next one work.*
>
> *Tools like Qlik sit on top of this same foundation. The dashboard Nick showed you earlier? It's powered by the same governed data. The agent I just showed you? Same data, same rules.*
>
> *If you're building AI on ungoverned data, you're building on sand. Get the foundation right — everything above it gets better."*

**Stop the Java streamer (Ctrl+C)**

---
## Cleanup (optional)

In [ ]:
-- DROP DATABASE AI_CAMP;
-- DROP ROLE SUMMIT_ANALYST;
-- DROP ROLE SUMMIT_STORE_MGR_PARK_CITY;